# Lab Day 1 — Lê Nguyễn Trâm Anh — 2A202602760
Notebook chạy theo stage trên Kaggle GPU. Mỗi run được lưu ngay; chạy lại cell sẽ resume nếu fingerprint cấu hình khớp. **Không xem eval trước Stage 5.**

In [ ]:
# Stage 0 — Bootstrap Kaggle/local
from pathlib import Path
import gzip, json, os, shutil, subprocess, sys, zipfile
IS_KAGGLE = Path('/kaggle/input').exists()
if IS_KAGGLE:
    candidates = list(Path('/kaggle/input').glob('**/data/covtype.csv.gz')) or list(Path('/kaggle/input').glob('**/data/covtype.csv'))
    if not candidates:
        bundles=list(Path('/kaggle/input').glob('**/kaggle_bundle_2A202602760.zip'))
        if not bundles: raise FileNotFoundError('Không thấy data/covtype.csv(.gz) hoặc kaggle_bundle_2A202602760.zip.')
        unpacked=Path('/kaggle/working/_input_bundle')
        with zipfile.ZipFile(bundles[0]) as zf: zf.extractall(unpacked)
        candidates=list(unpacked.glob('**/data/covtype.csv.gz')) or list(unpacked.glob('**/data/covtype.csv'))
        if not candidates: raise FileNotFoundError('Bundle không chứa data/covtype.csv(.gz).')
    source_root = candidates[0].parents[1]
    REPO_ROOT = Path('/kaggle/working/neuralnetwork_lab')
    if not REPO_ROOT.exists(): shutil.copytree(source_root, REPO_ROOT, ignore=shutil.ignore_patterns('.git','__pycache__','data/processed'))
else:
    here = Path.cwd().resolve()
    REPO_ROOT = next(p for p in [here,*here.parents] if (p/'data/covtype.csv.gz').exists() or (p/'data/covtype.csv').exists())
csv_path=REPO_ROOT/'data/covtype.csv'; gz_path=REPO_ROOT/'data/covtype.csv.gz'
if csv_path.exists() and not gz_path.exists():
    print('Creating writable covtype.csv.gz ...')
    with csv_path.open('rb') as src, gzip.open(gz_path,'wb') as dst: shutil.copyfileobj(src,dst)
SUB=REPO_ROOT/'submission_2A202602760'; RESULTS=SUB/'results'; FIGURES=SUB/'figures'; CHECKPOINTS=REPO_ROOT/'runtime_checkpoints'
for p in (SUB,RESULTS,FIGURES,CHECKPOINTS,SUB/'code'): p.mkdir(parents=True,exist_ok=True)
sys.path.insert(0,str(REPO_ROOT/'code')); os.chdir(REPO_ROOT)
if IS_KAGGLE and not any(RESULTS.glob('*.json')):
    archives=list(Path('/kaggle/input').glob('**/artifacts_stage*.zip'))
    if archives:
        def archive_score(path):
            with zipfile.ZipFile(path) as zf: return sum(n.startswith('results/') and n.endswith('.json') for n in zf.namelist())
        newest=max(archives,key=archive_score); restore=REPO_ROOT/'_restore'
        with zipfile.ZipFile(newest) as zf: zf.extractall(restore)
        for name,target in [('results',RESULTS),('figures',FIGURES),('runtime_checkpoints',CHECKPOINTS)]:
            src=restore/name
            if src.exists(): shutil.copytree(src,target,dirs_exist_ok=True)
        for name in ('health_checks.json','selection_state.json'):
            if (restore/name).exists(): shutil.copy2(restore/name,SUB/name)
import torch
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
if IS_KAGGLE and DEVICE!='cuda': raise RuntimeError('Settings → Accelerator → GPU')
print('repo=',REPO_ROOT,'device=',DEVICE,'torch=',torch.__version__)
if torch.cuda.is_available(): print('gpu=',torch.cuda.get_device_name(0))

## Stage 1 — Dữ liệu và health checks
Kỳ vọng: loss bước 0 gần `ln(7)`, logits `(B,7)`, đúng 47.879 tham số, mọi gradient khác 0 và overfit được 20 mẫu.

In [ ]:
from data import prepare_data
from workflow import run_health_checks,save_stage_archive
processed=REPO_ROOT/'data/processed'
if not (processed/'train.npz').exists(): subprocess.run([sys.executable,'scripts/split_data.py'],cwd=REPO_ROOT,check=True)
data=prepare_data(DEVICE,0.2,42,str(processed))
assert torch.allclose(data['X_tr'][:,:10].mean(0),torch.zeros(10,device=DEVICE),atol=2e-4)
assert torch.allclose(data['X_tr'][:,:10].std(0,unbiased=False),torch.ones(10,device=DEVICE),atol=2e-4)
health=run_health_checks(data,FIGURES/'health_overfit20.png')
(SUB/'health_checks.json').write_text(json.dumps(health,indent=2),encoding='utf-8')
display(health); print(save_stage_archive('stage1',SUB,CHECKPOINTS,REPO_ROOT))

## Stage 2 — Baseline, learning-rate sweep và nhiễu seed
Dự đoán: lr quá nhỏ hội tụ chậm, lr quá lớn dao động; chỉ chọn lr bằng validation. Ba seed dùng để tính `2σ`.

In [ ]:
from workflow import baseline_sweep_configs,official_baseline_configs,run_many,select_best,seed_statistics
sweep=run_many(baseline_sweep_configs(),data,RESULTS,FIGURES,CHECKPOINTS)
selected_lr=float(select_best(sweep)['cfg']['lr'])
baseline_runs=run_many(official_baseline_configs(selected_lr),data,RESULTS,FIGURES,CHECKPOINTS)
baseline_stats=seed_statistics(baseline_runs)
state={'selected_lr':selected_lr,'baseline_ids':[r['cfg']['exp_id'] for r in baseline_runs],'baseline_stats':baseline_stats}
(SUB/'selection_state.json').write_text(json.dumps(state,indent=2),encoding='utf-8')
display(state); print(save_stage_archive('stage2',SUB,CHECKPOINTS,REPO_ROOT))

## Stage 3 — Bảy nhóm thí nghiệm
Loss: CE được kỳ vọng ổn định hơn MSE; class weighting có thể giúp lớp hiếm. Optimizer chỉ được so sau khi mỗi họ được dò ba lr. Batch làm đổi số update; dropout chỉ hữu ích khi overfit. Clipping phải được kiểm chứng bằng grad norm ở lr cao. FP16 có thể giảm memory nhưng chưa chắc nhanh. Zeros dự kiến hỏng do đối xứng.

In [ ]:
from workflow import optimizer_configs,topic_configs,make_comparison_figures
state=json.loads((SUB/'selection_state.json').read_text(encoding='utf-8')); base_cfg=official_baseline_configs(state['selected_lr'])[0]
optimizer_runs=run_many(optimizer_configs(base_cfg),data,RESULTS,FIGURES,CHECKPOINTS)
base_s1=baseline_runs[0] if 'baseline_runs' in globals() else run_many([base_cfg],data,RESULTS,FIGURES,CHECKPOINTS)[0]
clip_c=float(base_s1['summary']['grad_norm_p75']); bf16_ok=bool(torch.cuda.is_available() and torch.cuda.is_bf16_supported())
groups=topic_configs(base_cfg,clip_c,bf16_ok)
for group_name,configs in groups.items():
    print('===',group_name,'==='); run_many(configs,data,RESULTS,FIGURES,CHECKPOINTS)
    print(save_stage_archive(f'stage3_{group_name}',SUB,CHECKPOINTS,REPO_ROOT))
make_comparison_figures(RESULTS,FIGURES)

## Stage 4 — Hai candidate, mỗi candidate ba seed
Candidate chỉ dùng validation. Nếu chênh lệch mean không vượt `max(2σ,0.002)`, ưu tiên ổn định, đơn giản và thời gian.

In [ ]:
from results_table import load_results
from workflow import build_candidate_configs,expand_candidate_seeds,choose_final_candidate
state=json.loads((SUB/'selection_state.json').read_text(encoding='utf-8')); all_results=load_results(RESULTS); base_cfg=official_baseline_configs(state['selected_lr'])[0]
candidate_a,candidate_b=build_candidate_configs(all_results,base_cfg,state['baseline_stats']['noise_2sigma'])
a_runs=run_many(expand_candidate_seeds(candidate_a,'a'),data,RESULTS,FIGURES,CHECKPOINTS)
b_runs=run_many(expand_candidate_seeds(candidate_b,'b'),data,RESULTS,FIGURES,CHECKPOINTS)
winner,chosen_runs,candidate_stats=choose_final_candidate(a_runs,b_runs,state['baseline_stats']['noise_2sigma'])
state.update({'winner':winner,'final_ids':[r['cfg']['exp_id'] for r in chosen_runs],'candidate_stats':candidate_stats})
(SUB/'selection_state.json').write_text(json.dumps(state,indent=2),encoding='utf-8'); make_comparison_figures(RESULTS,FIGURES)
display(state); print(save_stage_archive('stage4',SUB,CHECKPOINTS,REPO_ROOT))

## Stage 5 — Khóa lựa chọn rồi mới đánh giá eval
Cell tạo prediction baseline/final seed 1, gọi đúng `scripts/evaluate.py`, rồi sinh Excel và báo cáo. Không thay đổi cấu hình sau khi bắt đầu.

In [ ]:
from train import final_eval as write_eval_predictions
from reporting import build_xlsx,generate_report,read_json
state=json.loads((SUB/'selection_state.json').read_text(encoding='utf-8')); base_cfg=official_baseline_configs(state['selected_lr'])[0]
base_result=run_many([base_cfg],data,RESULTS,FIGURES,CHECKPOINTS)[0]
all_results=load_results(RESULTS); candidate_a,candidate_b=build_candidate_configs(all_results,base_cfg,state['baseline_stats']['noise_2sigma'])
winner_cfg=candidate_a if state['winner']=='a' else candidate_b; winner_cfg=expand_candidate_seeds(winner_cfg,state['winner'])[0]
final_result=run_many([winner_cfg],data,RESULTS,FIGURES,CHECKPOINTS)[0]
baseline_pred=REPO_ROOT/'baseline_predictions_eval.csv'; baseline_json=REPO_ROOT/'baseline_eval_result.json'
write_eval_predictions(base_result['cfg'],base_result,data,str(baseline_pred))
subprocess.run([sys.executable,'scripts/evaluate.py','--pred',str(baseline_pred),'--out',str(baseline_json)],cwd=REPO_ROOT,check=True)
write_eval_predictions(final_result['cfg'],final_result,data,str(SUB/'predictions_eval.csv'))
subprocess.run([sys.executable,'scripts/evaluate.py','--pred',str(SUB/'predictions_eval.csv'),'--out',str(SUB/'eval_result.json')],cwd=REPO_ROOT,check=True)
baseline_eval,final_scores=read_json(baseline_json),read_json(SUB/'eval_result.json'); all_results=load_results(RESULTS)
build_xlsx(all_results,REPO_ROOT/'templates/experiment_table_template.xlsx',SUB/'experiments.xlsx',state['baseline_ids'],state['final_ids'],baseline_eval,final_scores)
generate_report(all_results,read_json(SUB/'health_checks.json'),state['baseline_ids'],state['final_ids'],baseline_eval,final_scores,state['winner'],SUB/'REPORT.md')
print('FINAL EVAL:',final_scores['accuracy'],final_scores['macro_f1'])

## Stage 6 — Đồng bộ code, validate và đóng ZIP
ZIP chỉ được tạo khi hợp lệ; checkpoint và dữ liệu không được đưa vào bài nộp.

In [ ]:
for source in (REPO_ROOT/'code').iterdir():
    if source.is_file() and source.suffix in {'.py','.ipynb','.txt'}: shutil.copy2(source,SUB/'code'/source.name)
from validate_submission import validate_submission,create_zip
errors=validate_submission(SUB,REPO_ROOT/'data/processed/eval.npz')
if errors: raise AssertionError('SUBMISSION CHƯA HỢP LỆ:\n- '+'\n- '.join(errors))
zip_path=create_zip(SUB,REPO_ROOT/'submission_2A202602760.zip')
print('SUBMISSION HỢP LỆ:',zip_path); print('Tải ZIP từ Kaggle Output; không chỉnh cấu hình sau khi xem eval.')